In [1]:
import sys
sys.executable

'/Users/paigeblackstone/projects/f1-worldchamp/.venv/bin/python'

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/Users/paigeblackstone/Library/Mobile Documents/com~apple~CloudDocs/Formula 1 Fun")
csv_files = sorted(DATA_DIR.glob("*.csv"))
len(csv_files), [p.name for p in csv_files[:10]]


(14,
 ['circuits.csv',
  'constructor_results.csv',
  'constructor_standings.csv',
  'constructors.csv',
  'driver_standings.csv',
  'drivers.csv',
  'lap_times.csv',
  'pit_stops.csv',
  'qualifying.csv',
  'races.csv'])

In [3]:
tables = {p.stem.lower(): pd.read_csv(p) for p in csv_files}
sorted(tables.keys())

['circuits',
 'constructor_results',
 'constructor_standings',
 'constructors',
 'driver_standings',
 'drivers',
 'lap_times',
 'pit_stops',
 'qualifying',
 'races',
 'results',
 'seasons',
 'sprint_results',
 'status']

In [4]:
summary = (
    pd.DataFrame([{"table": k, "rows": v.shape[0], "cols": v.shape[1]} for k, v in tables.items()])
      .sort_values("rows", ascending=False)
      .reset_index(drop=True)
)
summary.head(15)

,table,rows,cols
0,lap_times,589081,6
1,driver_standings,34863,7
2,results,26759,18
3,constructor_standings,13391,7
4,constructor_results,12625,5
5,pit_stops,11371,7
6,qualifying,10494,9
7,races,1125,18
8,drivers,861,9
9,sprint_results,360,16


In [5]:
for name in ["results", "races", "drivers", "constructors", "circuits", "qualifying"]:
    print("\n", name.upper())
    print(tables[name].columns.tolist())


 RESULTS
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']

 RACES
['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time']

 DRIVERS
['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob', 'nationality', 'url']

 CONSTRUCTORS
['constructorId', 'constructorRef', 'name', 'nationality', 'url']

 CIRCUITS
['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt', 'url']

 QUALIFYING
['qualifyId', 'raceId', 'driverId', 'constructorId', 'number', 'position', 'q1', 'q2', 'q3']


In [6]:
assert tables["races"]["raceId"].is_unique
assert tables["drivers"]["driverId"].is_unique
assert tables["constructors"]["constructorId"].is_unique
assert tables["results"]["resultId"].is_unique

In [7]:
tables["results"].groupby(["raceId", "driverId"]).size().value_counts()

1    26583
2       79
3        6
Name: count, dtype: int64

In [13]:
set(tables["results"]["raceId"]) - set(tables["races"]["raceId"])
set(tables["results"]["raceId"]) - set(tables["drivers"]["driverId"])
set(tables["results"]["raceId"]) - set(tables["constructors"]["constructorId"])

{43,
 165,
 212,
 216,
 217,
 218,
 219,
 220,
 221,
 222,
 223,
 224,
 225,
 226,
 227,
 228,
 229,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 239,
 240,
 241,
 242,
 243,
 244,
 245,
 246,
 247,
 248,
 249,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 257,
 258,
 259,
 260,
 261,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 269,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 313,
 314,
 315,
 316,
 317,
 318,
 319,
 320,
 321,
 322,
 323,
 324,
 325,
 326,
 327,
 328,
 329,
 330,
 331,
 332,
 333,
 334,
 335,
 336,
 337,
 338,
 339,
 340,
 341,
 342,
 343,
 344,
 345,
 346,
 347,
 348,
 349,
 350,
 351,
 352,
 353,
 354,
 355,
 356,
 357,
 358,
 359,
 360,
 361,
 362,
 363,
 364,
 365,
 366,
 367,
 368,
 369,
 370,
 371,
 372,
 373,
 374,
 375,
 376,
 377,
 378,
 379,

In [15]:
results = tables["results"].copy()


In [16]:
results["target_points"] = (results["points"] > 0).astype(int)
results[["points", "target_points"]].head()


,points,target_points
0,10.0,1
1,8.0,1
2,6.0,1
3,5.0,1
4,4.0,1


In [17]:
results = tables["results"].copy()
results["target_points"] = (results["points"] > 0).astype(int)
results["target_points"].value_counts()


target_points
0    18589
1     8170
Name: count, dtype: int64

In [20]:
# --- Load base tables ---
results = tables["results"].copy()
races = tables["races"].copy()
drivers = tables["drivers"].copy()
constructors = tables["constructors"].copy()
circuits = tables["circuits"].copy()
status = tables["status"].copy()

# --- Parse race date ---
races["date"] = pd.to_datetime(races["date"])

# --- Build master (results spine) ---
master = (
    results
    .merge(races, on="raceId", how="left")
    .merge(drivers, on="driverId", how="left")
    .merge(constructors, on="constructorId", how="left")
    .merge(circuits, on="circuitId", how="left")
    .merge(status, on="statusId", how="left")
)

master.shape


MergeError: Passing 'suffixes' which cause duplicate columns {'url_x', 'url_y'} is not allowed.

In [18]:
qualifying = tables["qualifying"].copy()

master = (
    master
    .merge(
        qualifying[["raceId", "driverId", "position"]],
        on=["raceId", "driverId"],
        how="left"
    )
    .rename(columns={"position": "qualifying_position"})
)

master["qualifying_position"].describe()

NameError: name 'master' is not defined